# 🔍 Research Gap Finder — Google Colab
**Model:** Qwen2.5-7B-Instruct (4-bit)  |  **GPU:** T4

> ⚠️ Before running: **Runtime → Change runtime type → T4 GPU**

Run all cells top to bottom. The public link appears at the end.

In [ ]:
# Cell 1 — Check GPU
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout)
print('✅ GPU ready' if r.returncode == 0 else '❌ No GPU — change runtime type!')


In [ ]:
# Cell 2 — Install dependencies
!pip install -q streamlit pdfplumber pandas plotly reportlab \
    sentence-transformers faiss-cpu numpy \
    transformers accelerate bitsandbytes pyngrok
print('✅ All packages installed')


In [ ]:
# Cell 3 — pdf_reader.py
%%writefile pdf_reader.py
"""
Phase 1: PDF reading and text extraction
"""

import re
import pdfplumber


def extract_text_from_pdf(file_path: str) -> str:
    """Extract full text from a PDF file path"""
    full_text = []
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text.append(text)
    return "\n\n".join(full_text)


def extract_text_from_bytes(file_bytes: bytes) -> str:
    """Extract text from bytes (for use with Streamlit uploader)"""
    import io
    full_text = []
    with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text.append(text)
    return "\n\n".join(full_text)


def clean_text(raw_text: str) -> str:
    """Clean noise from extracted text"""
    text = re.sub(r'\n{3,}', '\n\n', raw_text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    lines = text.split('\n')
    cleaned_lines = [l for l in lines if len(l.strip()) > 3 or l.strip() == '']
    return '\n'.join(cleaned_lines).strip()


def truncate_for_api(text: str, max_chars: int = 12000) -> str:
    """Truncate text if too long for the API"""
    if len(text) <= max_chars:
        return text
    return text[:8000] + "\n\n[...content truncated...]\n\n" + text[-4000:]


def process_pdf(file_input, filename: str = "paper") -> dict:
    """
    Process a full PDF file and return cleaned data.
    file_input: either a str path or bytes
    """
    if isinstance(file_input, bytes):
        raw_text = extract_text_from_bytes(file_input)
    else:
        raw_text = extract_text_from_pdf(file_input)

    cleaned = clean_text(raw_text)
    truncated = truncate_for_api(cleaned)

    return {
        "filename": filename,
        "raw_length": len(raw_text),
        "cleaned_length": len(cleaned),
        "text": truncated,
        "status": "ok" if len(cleaned) > 200 else "too_short"
    }


if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1:
        result = process_pdf(sys.argv[1], sys.argv[1])
        print(f"File:           {result['filename']}")
        print(f"Raw length:     {result['raw_length']} chars")
        print(f"Cleaned length: {result['cleaned_length']} chars")
        print(f"Status:         {result['status']}")
        print("\nFirst 500 chars:")
        print(result['text'][:500])


In [ ]:
# Cell 4 — rag_pipeline.py
%%writefile rag_pipeline.py
"""
RAG Pipeline: Chunk → Embed → Index → Retrieve
Uses sentence-transformers for embeddings and FAISS for fast similarity search.
No external API needed — runs fully local.
"""

import re
import numpy as np
from typing import List, Dict
from sentence_transformers import SentenceTransformer
import faiss

# ── Embedding model (downloads once, ~90MB, runs on CPU or GPU) ──
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
_embed_model: SentenceTransformer | None = None


def get_embed_model() -> SentenceTransformer:
    """Lazy-load the embedding model (singleton)"""
    global _embed_model
    if _embed_model is None:
        _embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    return _embed_model


# ──────────────────────────────────────────────
# Step 1 — Chunking
# ──────────────────────────────────────────────

def chunk_text(
    text: str,
    chunk_size: int = 400,
    overlap: int = 80
) -> List[str]:
    """
    Split text into overlapping word-level chunks.
    chunk_size : target words per chunk
    overlap    : words shared between adjacent chunks
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        if len(chunk.strip()) > 50:          # skip tiny fragments
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


def chunk_papers(papers: List[Dict]) -> List[Dict]:
    """
    Chunk all papers and tag each chunk with its source filename.
    Returns a flat list of chunk dicts:
      { "filename", "chunk_id", "text" }
    """
    all_chunks = []
    for paper in papers:
        chunks = chunk_text(paper["text"])
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "filename": paper["filename"],
                "chunk_id": f"{paper['filename']}::{i}",
                "text": chunk
            })
    return all_chunks


# ──────────────────────────────────────────────
# Step 2 — Embedding & Indexing
# ──────────────────────────────────────────────

class PaperIndex:
    """
    FAISS vector index over paper chunks.
    Build once per session, query many times.
    """

    def __init__(self):
        self.chunks: List[Dict] = []
        self.index: faiss.Index | None = None
        self.embeddings: np.ndarray | None = None

    def build(self, chunks: List[Dict], batch_size: int = 64) -> None:
        """Embed all chunks and build the FAISS index"""
        self.chunks = chunks
        texts = [c["text"] for c in chunks]

        model = get_embed_model()
        embeddings = model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=False,
            normalize_embeddings=True   # cosine via inner-product
        )
        self.embeddings = embeddings.astype("float32")

        dim = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)   # inner-product = cosine (normalized)
        self.index.add(self.embeddings)

    def retrieve(self, query: str, top_k: int = 6) -> List[Dict]:
        """Return the top-k most relevant chunks for a query"""
        if self.index is None or len(self.chunks) == 0:
            return []

        model = get_embed_model()
        q_emb = model.encode(
            [query],
            normalize_embeddings=True,
            show_progress_bar=False
        ).astype("float32")

        scores, indices = self.index.search(q_emb, min(top_k, len(self.chunks)))

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= 0:
                chunk = dict(self.chunks[idx])
                chunk["score"] = float(score)
                results.append(chunk)
        return results

    def retrieve_for_paper(self, filename: str, query: str, top_k: int = 4) -> List[Dict]:
        """Retrieve top-k chunks from a specific paper only"""
        all_results = self.retrieve(query, top_k=top_k * 4)
        filtered = [r for r in all_results if r["filename"] == filename]
        return filtered[:top_k]

    @property
    def total_chunks(self) -> int:
        return len(self.chunks)

    @property
    def papers(self) -> List[str]:
        seen = []
        for c in self.chunks:
            if c["filename"] not in seen:
                seen.append(c["filename"])
        return seen


# ──────────────────────────────────────────────
# Step 3 — Context builder (for LLM prompts)
# ──────────────────────────────────────────────

def build_context(chunks: List[Dict], max_chars: int = 3000) -> str:
    """
    Concatenate retrieved chunks into a context string for the LLM.
    Respects max_chars to stay within the model's context window.
    """
    parts = []
    total = 0
    for chunk in chunks:
        snippet = chunk["text"].strip()
        if total + len(snippet) > max_chars:
            remaining = max_chars - total
            if remaining > 100:
                parts.append(snippet[:remaining] + "...")
            break
        parts.append(snippet)
        total += len(snippet)
    return "\n\n---\n\n".join(parts)


def build_paper_context(
    index: PaperIndex,
    filename: str,
    query: str,
    top_k: int = 4,
    max_chars: int = 3000
) -> str:
    """Retrieve and format context for a single paper"""
    chunks = index.retrieve_for_paper(filename, query, top_k=top_k)
    if not chunks:
        # fallback: retrieve globally
        chunks = index.retrieve(query, top_k=top_k)
        chunks = [c for c in chunks if c["filename"] == filename]
    return build_context(chunks, max_chars=max_chars)


def build_cross_paper_context(
    index: PaperIndex,
    query: str,
    top_k_per_paper: int = 2,
    max_chars: int = 4000
) -> str:
    """
    Retrieve chunks from EACH paper for cross-paper analysis (gap detection).
    Ensures every paper contributes at least one chunk.
    """
    all_chunks = []
    for filename in index.papers:
        chunks = index.retrieve_for_paper(filename, query, top_k=top_k_per_paper)
        all_chunks.extend(chunks)
    # sort by relevance score
    all_chunks.sort(key=lambda x: x.get("score", 0), reverse=True)
    return build_context(all_chunks, max_chars=max_chars)


if __name__ == "__main__":
    # Quick smoke test
    sample_papers = [
        {
            "filename": "paper1.pdf",
            "text": (
                "This paper proposes a CNN-based approach for car damage detection. "
                "We train on the CarDD dataset with 4000 labeled images. "
                "Our ResNet-50 model achieves 91% accuracy. "
                "Limitations include small dataset size and no night-time images. " * 20
            )
        },
        {
            "filename": "paper2.pdf",
            "text": (
                "We present an EfficientNet model for automated insurance claim processing. "
                "The system classifies damage severity into minor, moderate, and severe. "
                "Evaluated on a private dataset of 10,000 images. Accuracy: 88%. "
                "The model does not support video input or Arabic language reports. " * 20
            )
        }
    ]

    print("Chunking papers...")
    chunks = chunk_papers(sample_papers)
    print(f"Total chunks: {len(chunks)}")

    print("Building index...")
    idx = PaperIndex()
    idx.build(chunks)
    print(f"Index size: {idx.total_chunks} chunks")

    print("\nRetrieval test — query: 'dataset limitations'")
    results = idx.retrieve("dataset limitations", top_k=3)
    for r in results:
        print(f"  [{r['filename']}] score={r['score']:.3f} — {r['text'][:80]}...")

    print("\nCross-paper context for gap detection:")
    ctx = build_cross_paper_context(idx, "research gaps limitations future work")
    print(ctx[:400])


In [ ]:
# Cell 5 — analyzer.py
%%writefile analyzer.py
import json
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from rag_pipeline import PaperIndex, chunk_papers, build_paper_context, build_cross_paper_context

MODEL_ID   = "Qwen/Qwen2.5-7B-Instruct"
_tokenizer = None
_model     = None


def load_model():
    global _tokenizer, _model
    if _model is not None:
        return _tokenizer, _model
    print(f"Loading {MODEL_ID} with 4-bit quantization...")
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    _tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    _model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16
    )
    _model.eval()
    print("Model loaded.")
    return _tokenizer, _model


def _infer(prompt, max_new_tokens=800, temperature=0.1):
    tokenizer, model = load_model()
    msgs = [
        {"role": "system", "content": "You are an expert research analyst. Always respond with valid JSON only."},
        {"role": "user",   "content": prompt}
    ]
    text   = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            pad_token_id=tokenizer.eos_token_id
        )
    generated = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def _parse_json(raw):
    raw = re.sub(r"^```json\s*", "", raw.strip())
    raw = re.sub(r"^```\s*",     "", raw)
    raw = re.sub(r"\s*```$",     "", raw)
    m = re.search(r"\{[\s\S]*\}", raw)
    if m:
        raw = m.group(0)
    return json.loads(raw)


def build_index(papers):
    chunks = chunk_papers(papers)
    index  = PaperIndex()
    index.build(chunks)
    return index


SUMMARY_PROMPT = """Analyze the research paper excerpt below. Return valid JSON only, no extra text.

{{
  "title": "Paper title",
  "year": "year or unknown",
  "authors": "authors or unknown",
  "problem": "one-sentence problem statement",
  "method": "method or algorithm used",
  "dataset": "dataset(s) used",
  "main_result": "key result",
  "limitations": ["limitation 1", "limitation 2", "limitation 3"],
  "keywords": ["kw1", "kw2", "kw3", "kw4", "kw5"]
}}

--- PAPER EXCERPT ---
{context}
--- END ---"""


def summarize_paper(index, filename):
    context = build_paper_context(
        index, filename,
        query="title authors abstract problem method dataset results limitations keywords",
        top_k=4,
        max_chars=2500
    )
    if not context.strip():
        return {
            "filename": filename, "status": "error", "error": "No RAG content",
            "title": filename, "year": "?", "authors": "?", "problem": "?",
            "method": "?", "dataset": "?", "main_result": "?",
            "limitations": [], "keywords": []
        }
    try:
        raw    = _infer(SUMMARY_PROMPT.format(context=context), max_new_tokens=600)
        result = _parse_json(raw)
        if isinstance(result, dict):
            result["filename"] = filename
            result["status"]   = "ok"
            return result
        raise ValueError("Response is not a dict")
    except Exception as e:
        return {
            "filename": filename, "status": "error", "error": str(e),
            "title": filename, "year": "?", "authors": "?", "problem": "?",
            "method": "?", "dataset": "?", "main_result": "?",
            "limitations": [], "keywords": []
        }


GAP_PROMPT = """Identify research gaps from {n_papers} research paper excerpts below.
Return valid JSON only, no extra text.

{{
  "common_methods": ["method1", "method2", "method3"],
  "common_datasets": ["dataset1", "dataset2"],
  "common_limitations": ["limitation1", "limitation2", "limitation3"],
  "research_gaps": [
    {{"gap": "gap description", "evidence": "evidence from papers", "novelty_score": 8.5}},
    {{"gap": "gap 2",           "evidence": "evidence",             "novelty_score": 7.0}},
    {{"gap": "gap 3",           "evidence": "evidence",             "novelty_score": 6.0}}
  ],
  "suggested_ideas": [
    {{"idea": "idea title", "addresses_gap": "gap it targets", "feasibility": "High",   "why_promising": "reason"}},
    {{"idea": "idea 2",     "addresses_gap": "gap it targets", "feasibility": "Medium", "why_promising": "reason"}}
  ],
  "overall_summary": "2-3 sentence overview of the field"
}}

--- CROSS-PAPER EXCERPTS ---
{context}

--- PAPER SUMMARIES ---
{summaries_text}"""


def _build_summaries_text(summaries):
    lines = []
    for i, s in enumerate(summaries, 1):
        title    = s.get("title",    s.get("filename", f"Paper {i}"))
        method   = s.get("method",   "N/A")
        dataset  = s.get("dataset",  "N/A")
        lims     = s.get("limitations", [])
        lims_str = "; ".join(lims) if isinstance(lims, list) else str(lims)
        lines.append(f"Paper {i}: {title}")
        lines.append(f"  Method: {method}")
        lines.append(f"  Dataset: {dataset}")
        lines.append(f"  Limitations: {lims_str}")
        lines.append("")
    return "\n".join(lines)


def detect_gaps(summaries, index):
    context = build_cross_paper_context(
        index,
        query="research gaps limitations future work unexplored directions",
        top_k_per_paper=2,
        max_chars=3000
    )
    prompt = GAP_PROMPT.format(
        n_papers=len(summaries),
        context=context,
        summaries_text=_build_summaries_text(summaries)
    )
    try:
        raw    = _infer(prompt, max_new_tokens=900, temperature=0.2)
        result = _parse_json(raw)
        if isinstance(result, dict):
            result["status"] = "ok"
            return result
        raise ValueError("Response is not a dict")
    except Exception as e:
        return {
            "status": "error", "error": str(e),
            "common_methods": [], "common_datasets": [], "common_limitations": [],
            "research_gaps": [], "suggested_ideas": [],
            "overall_summary": "Analysis failed."
        }


In [ ]:
# Cell 6 — report.py
%%writefile report.py
"""
Phase 4: Professional PDF report generation
"""

import io
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    HRFlowable, KeepTogether
)
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT

# Project colors
BLUE_DARK   = colors.HexColor("#0C447C")
BLUE_MID    = colors.HexColor("#185FA5")
BLUE_LIGHT  = colors.HexColor("#E6F1FB")
TEAL        = colors.HexColor("#0F6E56")
TEAL_LIGHT  = colors.HexColor("#E1F5EE")
AMBER       = colors.HexColor("#854F0B")
AMBER_LIGHT = colors.HexColor("#FAEEDA")
GRAY_DARK   = colors.HexColor("#444441")
GRAY_LIGHT  = colors.HexColor("#F1EFE8")
WHITE       = colors.white


def build_styles():
    base = getSampleStyleSheet()

    title_style = ParagraphStyle(
        'ReportTitle', fontSize=22, textColor=WHITE,
        alignment=TA_CENTER, spaceAfter=6, fontName='Helvetica-Bold'
    )
    h1 = ParagraphStyle(
        'H1', fontSize=14, textColor=BLUE_DARK,
        spaceBefore=14, spaceAfter=6, fontName='Helvetica-Bold'
    )
    h2 = ParagraphStyle(
        'H2', fontSize=12, textColor=TEAL,
        spaceBefore=10, spaceAfter=4, fontName='Helvetica-Bold'
    )
    body = ParagraphStyle(
        'Body', fontSize=10, textColor=GRAY_DARK,
        spaceAfter=4, leading=15, fontName='Helvetica'
    )
    small = ParagraphStyle(
        'Small', fontSize=9, textColor=colors.HexColor("#888780"),
        spaceAfter=2, fontName='Helvetica'
    )
    gap_style = ParagraphStyle(
        'Gap', fontSize=10, textColor=BLUE_DARK,
        spaceBefore=4, spaceAfter=4, leftIndent=12, fontName='Helvetica-Bold'
    )
    idea_style = ParagraphStyle(
        'Idea', fontSize=10, textColor=TEAL,
        spaceBefore=4, spaceAfter=4, leftIndent=12, fontName='Helvetica-Bold'
    )
    return {
        'title': title_style, 'h1': h1, 'h2': h2,
        'body': body, 'small': small, 'gap': gap_style, 'idea': idea_style
    }


def score_color(score: float):
    if score >= 8:
        return TEAL
    elif score >= 6:
        return BLUE_MID
    else:
        return AMBER


def generate_report(summaries: list[dict], gaps: dict) -> bytes:
    """Generate a full PDF report and return it as bytes"""
    buffer = io.BytesIO()
    doc = SimpleDocTemplate(
        buffer, pagesize=A4,
        rightMargin=2*cm, leftMargin=2*cm,
        topMargin=2*cm, bottomMargin=2*cm
    )

    styles = build_styles()
    story = []

    # Cover
    story.append(Spacer(1, 1*cm))
    cover_data = [[Paragraph("Research Gap Finder", styles['title'])]]
    cover_table = Table(cover_data, colWidths=[17*cm])
    cover_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), BLUE_DARK),
        ('ROUNDEDCORNERS', [8]),
        ('TOPPADDING', (0,0), (-1,-1), 18),
        ('BOTTOMPADDING', (0,0), (-1,-1), 18),
        ('LEFTPADDING', (0,0), (-1,-1), 20),
        ('RIGHTPADDING', (0,0), (-1,-1), 20),
    ]))
    story.append(cover_table)
    story.append(Spacer(1, 0.4*cm))

    date_str = datetime.now().strftime("%Y-%m-%d")
    meta = Table([
        [Paragraph(f"Analysis date: {date_str}", styles['small']),
         Paragraph(f"Papers analyzed: {len(summaries)}", styles['small'])]
    ], colWidths=[8.5*cm, 8.5*cm])
    meta.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), GRAY_LIGHT),
        ('TOPPADDING', (0,0), (-1,-1), 6),
        ('BOTTOMPADDING', (0,0), (-1,-1), 6),
        ('LEFTPADDING', (0,0), (-1,-1), 10),
    ]))
    story.append(meta)
    story.append(Spacer(1, 0.6*cm))

    # Overall summary
    if gaps.get("overall_summary"):
        story.append(HRFlowable(width="100%", thickness=0.5, color=BLUE_LIGHT))
        story.append(Spacer(1, 0.2*cm))
        story.append(Paragraph("Overall Summary", styles['h1']))
        story.append(Paragraph(gaps["overall_summary"], styles['body']))
        story.append(Spacer(1, 0.3*cm))

    # Section 1: Paper Summaries
    story.append(HRFlowable(width="100%", thickness=1, color=BLUE_MID))
    story.append(Spacer(1, 0.2*cm))
    story.append(Paragraph("1. Paper Summaries", styles['h1']))
    story.append(Spacer(1, 0.2*cm))

    for i, s in enumerate(summaries, 1):
        title = s.get('title', s.get('filename', f'Paper {i}'))
        paper_block = Table([[Paragraph(f"#{i} — {title}", styles['h2'])]], colWidths=[17*cm])
        paper_block.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,-1), TEAL_LIGHT),
            ('TOPPADDING', (0,0), (-1,-1), 6),
            ('BOTTOMPADDING', (0,0), (-1,-1), 6),
            ('LEFTPADDING', (0,0), (-1,-1), 10),
            ('ROUNDEDCORNERS', [4]),
        ]))
        story.append(KeepTogether([paper_block]))
        story.append(Spacer(1, 0.15*cm))

        lims = s.get('limitations', [])
        lims_str = " | ".join(lims) if isinstance(lims, list) else str(lims) or "—"

        details = [
            ["Problem",     s.get('problem', '—')],
            ["Method",      s.get('method', '—')],
            ["Dataset",     s.get('dataset', '—')],
            ["Result",      s.get('main_result', '—')],
            ["Limitations", lims_str or "—"],
        ]
        detail_table = Table(
            [[Paragraph(k, styles['small']), Paragraph(v, styles['body'])] for k, v in details],
            colWidths=[3*cm, 14*cm]
        )
        detail_table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (0,-1), GRAY_LIGHT),
            ('GRID', (0,0), (-1,-1), 0.3, colors.HexColor("#D3D1C7")),
            ('TOPPADDING', (0,0), (-1,-1), 4),
            ('BOTTOMPADDING', (0,0), (-1,-1), 4),
            ('LEFTPADDING', (0,0), (-1,-1), 8),
            ('VALIGN', (0,0), (-1,-1), 'TOP'),
        ]))
        story.append(detail_table)
        story.append(Spacer(1, 0.4*cm))

    # Section 2: Comparison Matrix
    story.append(HRFlowable(width="100%", thickness=1, color=BLUE_MID))
    story.append(Spacer(1, 0.2*cm))
    story.append(Paragraph("2. Comparison Matrix", styles['h1']))
    story.append(Spacer(1, 0.2*cm))

    matrix_data = [["#", "Method", "Dataset", "Result"]]
    for i, s in enumerate(summaries, 1):
        matrix_data.append([
            str(i),
            s.get('method', '—')[:40],
            s.get('dataset', '—')[:30],
            s.get('main_result', '—')[:40],
        ])

    matrix_table = Table(matrix_data, colWidths=[0.8*cm, 5.5*cm, 4.5*cm, 6.2*cm])
    matrix_style = [
        ('BACKGROUND', (0,0), (-1,0), BLUE_DARK),
        ('TEXTCOLOR', (0,0), (-1,0), WHITE),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0), (-1,-1), 9),
        ('GRID', (0,0), (-1,-1), 0.3, colors.HexColor("#D3D1C7")),
        ('TOPPADDING', (0,0), (-1,-1), 5),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5),
        ('LEFTPADDING', (0,0), (-1,-1), 6),
        ('VALIGN', (0,0), (-1,-1), 'TOP'),
    ]
    for row_idx in range(1, len(matrix_data), 2):
        matrix_style.append(('BACKGROUND', (0,row_idx), (-1,row_idx), BLUE_LIGHT))
    matrix_table.setStyle(TableStyle(matrix_style))
    story.append(matrix_table)
    story.append(Spacer(1, 0.5*cm))

    # Section 3: Common Patterns
    story.append(HRFlowable(width="100%", thickness=1, color=BLUE_MID))
    story.append(Spacer(1, 0.2*cm))
    story.append(Paragraph("3. Common Patterns in the Field", styles['h1']))

    def render_tag_list(items: list):
        if not items:
            return Paragraph("No data", styles['body'])
        return Paragraph("  ·  ".join(items), styles['body'])

    common_data = [
        [Paragraph("Common Methods", styles['h2']),
         Paragraph("Common Datasets", styles['h2']),
         Paragraph("Common Limitations", styles['h2'])],
        [render_tag_list(gaps.get('common_methods', [])),
         render_tag_list(gaps.get('common_datasets', [])),
         render_tag_list(gaps.get('common_limitations', []))],
    ]
    common_table = Table(common_data, colWidths=[5.5*cm, 5.5*cm, 6*cm])
    common_table.setStyle(TableStyle([
        ('GRID', (0,0), (-1,-1), 0.3, colors.HexColor("#D3D1C7")),
        ('BACKGROUND', (0,0), (-1,0), GRAY_LIGHT),
        ('TOPPADDING', (0,0), (-1,-1), 6),
        ('BOTTOMPADDING', (0,0), (-1,-1), 6),
        ('LEFTPADDING', (0,0), (-1,-1), 8),
        ('VALIGN', (0,0), (-1,-1), 'TOP'),
    ]))
    story.append(common_table)
    story.append(Spacer(1, 0.5*cm))

    # Section 4: Research Gaps
    story.append(HRFlowable(width="100%", thickness=1, color=BLUE_MID))
    story.append(Spacer(1, 0.2*cm))
    story.append(Paragraph("4. Discovered Research Gaps", styles['h1']))
    story.append(Spacer(1, 0.2*cm))

    for i, gap in enumerate(gaps.get('research_gaps', []), 1):
        score = gap.get('novelty_score', 0)
        sc = score_color(score)

        gap_header = [[
            Paragraph(f"Gap #{i}", styles['gap']),
            Paragraph(f"Novelty: {score}/10", ParagraphStyle(
                'Score', fontSize=11, textColor=sc,
                alignment=TA_RIGHT, fontName='Helvetica-Bold'
            ))
        ]]
        gap_header_table = Table(gap_header, colWidths=[13*cm, 4*cm])
        gap_header_table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,-1), BLUE_LIGHT),
            ('TOPPADDING', (0,0), (-1,-1), 6),
            ('BOTTOMPADDING', (0,0), (-1,-1), 6),
            ('LEFTPADDING', (0,0), (-1,-1), 10),
        ]))
        story.append(gap_header_table)

        gap_detail = Table([
            [Paragraph("Gap:", styles['small']),
             Paragraph(gap.get('gap', '—'), styles['body'])],
            [Paragraph("Evidence:", styles['small']),
             Paragraph(gap.get('evidence', '—'), styles['body'])],
        ], colWidths=[2*cm, 15*cm])
        gap_detail.setStyle(TableStyle([
            ('GRID', (0,0), (-1,-1), 0.3, colors.HexColor("#D3D1C7")),
            ('TOPPADDING', (0,0), (-1,-1), 5),
            ('BOTTOMPADDING', (0,0), (-1,-1), 5),
            ('LEFTPADDING', (0,0), (-1,-1), 8),
            ('VALIGN', (0,0), (-1,-1), 'TOP'),
        ]))
        story.append(gap_detail)
        story.append(Spacer(1, 0.3*cm))

    # Section 5: Suggested Ideas
    story.append(HRFlowable(width="100%", thickness=1, color=TEAL))
    story.append(Spacer(1, 0.2*cm))
    story.append(Paragraph("5. Suggested Research Ideas", styles['h1']))
    story.append(Spacer(1, 0.2*cm))

    for i, idea in enumerate(gaps.get('suggested_ideas', []), 1):
        feasibility = idea.get('feasibility', '—')
        f_color = TEAL if 'High' in feasibility else (BLUE_MID if 'Medium' in feasibility else AMBER)

        idea_header = [[Paragraph(f"Idea #{i}: {idea.get('idea', '—')}", styles['idea'])]]
        idea_header_t = Table(idea_header, colWidths=[17*cm])
        idea_header_t.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,-1), TEAL_LIGHT),
            ('TOPPADDING', (0,0), (-1,-1), 6),
            ('BOTTOMPADDING', (0,0), (-1,-1), 6),
            ('LEFTPADDING', (0,0), (-1,-1), 10),
        ]))
        story.append(idea_header_t)

        idea_detail = Table([
            [Paragraph("Addresses gap:", styles['small']),
             Paragraph(idea.get('addresses_gap', '—'), styles['body'])],
            [Paragraph("Feasibility:", styles['small']),
             Paragraph(feasibility, ParagraphStyle('Feas', fontSize=10, textColor=f_color, fontName='Helvetica-Bold'))],
            [Paragraph("Why promising:", styles['small']),
             Paragraph(idea.get('why_promising', '—'), styles['body'])],
        ], colWidths=[2.5*cm, 14.5*cm])
        idea_detail.setStyle(TableStyle([
            ('GRID', (0,0), (-1,-1), 0.3, colors.HexColor("#D3D1C7")),
            ('TOPPADDING', (0,0), (-1,-1), 5),
            ('BOTTOMPADDING', (0,0), (-1,-1), 5),
            ('LEFTPADDING', (0,0), (-1,-1), 8),
            ('VALIGN', (0,0), (-1,-1), 'TOP'),
        ]))
        story.append(idea_detail)
        story.append(Spacer(1, 0.35*cm))

    # Footer
    story.append(Spacer(1, 0.5*cm))
    story.append(HRFlowable(width="100%", thickness=0.5, color=GRAY_LIGHT))
    story.append(Spacer(1, 0.2*cm))
    story.append(Paragraph(
        f"Generated by Research Gap Finder — {date_str}",
        ParagraphStyle('Footer', fontSize=8, textColor=colors.HexColor("#888780"),
                       alignment=TA_CENTER, fontName='Helvetica')
    ))

    doc.build(story)
    buffer.seek(0)
    return buffer.read()


if __name__ == "__main__":
    test_summaries = [
        {
            "filename": "paper1.pdf",
            "title": "Car Damage Detection using CNN",
            "year": "2023",
            "authors": "Ahmed et al.",
            "problem": "Automatic detection of car body damage from images",
            "method": "ResNet-50 with transfer learning",
            "dataset": "CarDD Dataset (4000 images)",
            "main_result": "91% accuracy on test set",
            "limitations": ["Small dataset", "Daylight images only", "No video support"],
            "keywords": ["CNN", "damage", "car", "ResNet", "detection"],
            "status": "ok"
        },
        {
            "filename": "paper2.pdf",
            "title": "Insurance Claim Automation via Deep Learning",
            "year": "2022",
            "authors": "Wang et al.",
            "problem": "Automating insurance damage assessment",
            "method": "EfficientNet + severity classifier",
            "dataset": "Private insurance dataset",
            "main_result": "Reduces assessment time by 70%",
            "limitations": ["Private data", "English only", "No multilingual support"],
            "keywords": ["insurance", "EfficientNet", "severity", "automation"],
            "status": "ok"
        }
    ]
    test_gaps = {
        "status": "ok",
        "common_methods": ["CNN", "ResNet", "EfficientNet"],
        "common_datasets": ["CarDD", "Private datasets"],
        "common_limitations": ["Small datasets", "No video support", "English only"],
        "research_gaps": [
            {
                "gap": "No studies support multilingual damage assessment",
                "evidence": "All studies use English-only datasets",
                "novelty_score": 9.0
            },
            {
                "gap": "No real-time video-based damage detection models",
                "evidence": "All studies rely on static images only",
                "novelty_score": 7.5
            }
        ],
        "suggested_ideas": [
            {
                "idea": "Multilingual car damage assessment system",
                "addresses_gap": "Lack of multilingual support",
                "feasibility": "High",
                "why_promising": "Large untapped non-English market"
            }
        ],
        "overall_summary": "The field is growing rapidly with a focus on static images. The biggest gap is multilingual support and video-based analysis."
    }

    pdf_bytes = generate_report(test_summaries, test_gaps)
    with open("/tmp/test_report.pdf", "wb") as f:
        f.write(pdf_bytes)
    print(f"Report generated: {len(pdf_bytes)} bytes")


In [ ]:
# Cell 7 — app.py
%%writefile app.py
import json
import streamlit as st
import plotly.express as px
from pdf_reader import process_pdf
from analyzer import build_index, summarize_paper, detect_gaps, MODEL_ID, load_model
from report import generate_report

st.set_page_config(page_title="Research Gap Finder", page_icon="🔍", layout="wide")

st.markdown("""
<style>
.main-header {
    background: linear-gradient(135deg, #0C447C, #185FA5);
    color: white; padding: 1.5rem 2rem;
    border-radius: 12px; margin-bottom: 1.5rem;
}
.gap-card {
    background: #EAF3DE; border: 1px solid #C0DD97;
    border-radius: 8px; padding: 1rem; margin: 0.5rem 0;
}
.idea-card {
    background: #EEEDFE; border: 1px solid #CECBF6;
    border-radius: 8px; padding: 1rem; margin: 0.5rem 0;
}
.score-high   { color: #0F6E56; font-weight: bold; font-size: 1.2em; }
.score-medium { color: #185FA5; font-weight: bold; font-size: 1.2em; }
.score-low    { color: #854F0B; font-weight: bold; font-size: 1.2em; }
</style>
""", unsafe_allow_html=True)

# ── Auto-load model on startup ──
if "model_loaded" not in st.session_state:
    st.session_state["model_loaded"] = False

if not st.session_state["model_loaded"]:
    with st.spinner("⏳ Loading Qwen2.5-7B... (3–5 min first time)"):
        load_model()
        st.session_state["model_loaded"] = True

# ── Sidebar ──
with st.sidebar:
    st.markdown("## ⚙️ Settings")
    st.markdown(f"**Model:** `{MODEL_ID}`")
    st.markdown("**Runtime:** Google Colab T4 GPU")
    st.success("✅ Qwen2.5-7B ready")
    st.markdown("---")
    st.markdown("### How to use")
    st.markdown("1. Upload 2–10 PDF papers\n2. Click **Start Analysis**\n3. View results & download report")
    st.markdown("---")
    st.caption("PDF → chunks → MiniLM → FAISS → RAG → Qwen2.5-7B → report")

# ── Header ──
st.markdown("""
<div class="main-header">
    <h1 style="margin:0; font-size:1.8rem;">🔍 Research Gap Finder</h1>
    <p style="margin:0.3rem 0 0; opacity:0.85;">
        Powered by Qwen2.5-7B + RAG — fully local on Colab T4
    </p>
</div>
""", unsafe_allow_html=True)

# ── File upload ──
st.markdown("### 📂 Upload Research Papers")
uploaded_files = st.file_uploader(
    "Select 2–10 PDF files",
    type=["pdf"],
    accept_multiple_files=True
)

if uploaded_files:
    n = len(uploaded_files)
    if n < 2:
        st.warning("⚠️ Upload at least 2 papers.")
    elif n > 10:
        st.warning("⚠️ Maximum 10 papers.")
    else:
        st.success(f"✅ {n} papers ready")
        cols = st.columns(min(n, 5))
        for i, f in enumerate(uploaded_files):
            with cols[i % 5]:
                st.caption(f"📄 {f.name[:22]}...")

# ── Analyze button ──
st.markdown("---")
files_ok    = bool(uploaded_files) and 2 <= len(uploaded_files) <= 10
analyze_btn = st.button(
    "🚀 Start Analysis",
    type="primary",
    disabled=not files_ok,
    use_container_width=True
)

# ── Analysis pipeline ──
if analyze_btn:

    with st.status("📖 Reading PDFs...", expanded=True) as status:
        papers   = []
        progress = st.progress(0)
        for i, f in enumerate(uploaded_files):
            p = process_pdf(f.read(), f.name)
            if p["status"] == "too_short":
                st.warning(f"⚠️ {f.name}: too short or scanned — skipped.")
            else:
                papers.append(p)
                st.write(f"✅ {f.name} — {p['cleaned_length']:,} chars")
            progress.progress((i + 1) / len(uploaded_files))
        status.update(label=f"✅ Loaded {len(papers)} papers", state="complete")

    if len(papers) < 2:
        st.error("❌ Need at least 2 valid papers.")
        st.stop()

    with st.status("🔢 Building RAG index...", expanded=True) as status:
        st.write("Chunking and embedding with MiniLM...")
        index = build_index(papers)
        st.write(f"✅ {index.total_chunks} chunks indexed")
        status.update(label="✅ RAG index ready", state="complete")

    with st.status("🧠 Summarizing papers...", expanded=True) as status:
        summaries = []
        p2        = st.progress(0)
        for i, paper in enumerate(papers):
            st.write(f"Analyzing: {paper['filename']}...")
            s = summarize_paper(index, paper["filename"])
            summaries.append(s)
            st.write(f"✅ {s.get('title', paper['filename'])[:55]}")
            p2.progress((i + 1) / len(papers))
        status.update(label=f"✅ Summarized {len(summaries)} papers", state="complete")

    with st.status("🔍 Detecting research gaps...", expanded=True) as status:
        st.write("Analyzing cross-paper patterns...")
        gaps   = detect_gaps(summaries, index)
        n_gaps = len(gaps.get("research_gaps", []))
        n_ideas= len(gaps.get("suggested_ideas", []))
        if gaps["status"] == "ok":
            status.update(label=f"✅ Found {n_gaps} gaps, {n_ideas} ideas", state="complete")
        else:
            st.warning(gaps.get("error", ""))
            status.update(label="⚠️ Done with warnings", state="complete")

    st.session_state["summaries"] = summaries
    st.session_state["gaps"]      = gaps
    st.rerun()

# ── Results ──
if "summaries" in st.session_state and "gaps" in st.session_state:
    summaries = st.session_state["summaries"]
    gaps      = st.session_state["gaps"]

    st.markdown("---")
    st.markdown("## 📊 Results")

    c1, c2, c3, c4 = st.columns(4)
    with c1: st.metric("Papers",      len(summaries))
    with c2: st.metric("Gaps Found",  len(gaps.get("research_gaps", [])))
    with c3: st.metric("Ideas",       len(gaps.get("suggested_ideas", [])))
    with c4:
        rg  = gaps.get("research_gaps", [])
        avg = sum(g.get("novelty_score", 0) for g in rg) / len(rg) if rg else 0
        st.metric("Avg Novelty", f"{avg:.1f}/10")

    st.markdown("---")
    tab1, tab2, tab3 = st.tabs([
        "📚 Paper Summaries",
        "🕳️ Research Gaps",
        "💡 Suggested Ideas"
    ])

    # Tab 1 — Summaries
    with tab1:
        st.markdown("### Paper Summaries")
        for i, s in enumerate(summaries, 1):
            title = s.get("title", s.get("filename", f"Paper {i}"))
            with st.expander(f"📄 {i}. {title}", expanded=(i == 1)):
                ca, cb = st.columns(2)
                with ca:
                    st.markdown(f"**Problem:** {s.get('problem', '—')}")
                    st.markdown(f"**Method:** {s.get('method', '—')}")
                    st.markdown(f"**Dataset:** {s.get('dataset', '—')}")
                with cb:
                    st.markdown(f"**Result:** {s.get('main_result', '—')}")
                    st.markdown(f"**Authors:** {s.get('authors', '—')} ({s.get('year', '—')})")
                lims = s.get("limitations", [])
                if lims:
                    st.markdown("**Limitations:**")
                    lims_list = lims if isinstance(lims, list) else [str(lims)]
                    for lim in lims_list:
                        st.markdown(f"  - {lim}")

    # Tab 2 — Research Gaps
    with tab2:
        st.markdown("### 🔴 Discovered Research Gaps")
        if gaps.get("overall_summary"):
            st.info(f"**Overview:** {gaps['overall_summary']}")

        ca, cb, cc = st.columns(3)
        with ca:
            st.markdown("**Common Methods**")
            for m in gaps.get("common_methods", []):
                st.markdown(f"- {m}")
        with cb:
            st.markdown("**Common Datasets**")
            for d in gaps.get("common_datasets", []):
                st.markdown(f"- {d}")
        with cc:
            st.markdown("**Common Limitations**")
            for lim in gaps.get("common_limitations", []):
                st.markdown(f"- {lim}")

        st.markdown("---")
        rg_sorted = sorted(
            gaps.get("research_gaps", []),
            key=lambda x: x.get("novelty_score", 0),
            reverse=True
        )
        for i, gap in enumerate(rg_sorted, 1):
            score = gap.get("novelty_score", 0)
            sc    = "score-high" if score >= 8 else ("score-medium" if score >= 6 else "score-low")
            st.markdown(f"""
            <div class="gap-card">
                <div style="display:flex; justify-content:space-between; align-items:center">
                    <b>Gap #{i}</b><span class="{sc}">Novelty: {score}/10</span>
                </div>
                <p style="margin:0.5rem 0 0">{gap.get('gap', '—')}</p>
                <p style="font-size:0.85em; color:#555; margin:0.3rem 0 0">📎 {gap.get('evidence', '—')}</p>
            </div>""", unsafe_allow_html=True)

        if rg_sorted:
            score_df = [{"Gap": f"Gap #{i+1}", "Novelty Score": g.get("novelty_score", 0)}
                        for i, g in enumerate(rg_sorted)]
            import pandas as pd
            fig = px.bar(
                pd.DataFrame(score_df), x="Gap", y="Novelty Score",
                title="Novelty Score Comparison",
                color="Novelty Score",
                color_continuous_scale=[[0, "#FAEEDA"], [0.5, "#185FA5"], [1, "#0F6E56"]],
                range_y=[0, 10]
            )
            fig.add_hline(y=7, line_dash="dash", line_color="#854F0B",
                          annotation_text="Important threshold (7+)")
            fig.update_layout(height=300)
            st.plotly_chart(fig, use_container_width=True)

    # Tab 3 — Ideas
    with tab3:
        st.markdown("### 💡 Suggested Research Ideas")
        ideas = gaps.get("suggested_ideas", [])
        if not ideas:
            st.warning("No ideas generated.")
        for i, idea in enumerate(ideas, 1):
            feas = idea.get("feasibility", "—")
            fc   = "#0F6E56" if "High" in feas else ("#185FA5" if "Medium" in feas else "#854F0B")
            st.markdown(f"""
            <div class="idea-card">
                <b>💡 Idea #{i}: {idea.get('idea', '—')}</b><br><br>
                <span>📌 Addresses: {idea.get('addresses_gap', '—')}</span><br>
                <span style="color:{fc}; font-weight:bold">⚡ Feasibility: {feas}</span><br>
                <span style="color:#555">🌟 {idea.get('why_promising', '—')}</span>
            </div>""", unsafe_allow_html=True)

    # ── Download ──
    st.markdown("---")
    st.markdown("### 📥 Download Report")
    _, col, _ = st.columns([1, 2, 1])
    with col:
        with st.spinner("Generating PDF..."):
            try:
                pdf = generate_report(summaries, gaps)
                st.download_button(
                    "⬇️ Download PDF Report", pdf,
                    "research_gap_report.pdf", "application/pdf",
                    use_container_width=True, type="primary"
                )
            except Exception as e:
                st.error(f"PDF error: {e}")

        st.download_button(
            "⬇️ Download JSON",
            json.dumps({"summaries": summaries, "gaps": gaps}, ensure_ascii=False, indent=2),
            "data.json", "application/json",
            use_container_width=True
        )

    st.markdown("---")
    if st.button("🔄 New Analysis", use_container_width=True):
        for k in ["summaries", "gaps"]:
            st.session_state.pop(k, None)
        st.rerun()


In [ ]:
# Cell 8 — Launch app via ngrok
NGROK_TOKEN = ''  # <-- paste your token from https://dashboard.ngrok.com

import subprocess, threading, time
from pyngrok import ngrok, conf

if not NGROK_TOKEN:
    print('⚠️  Add your ngrok token above!')
    print('Get it free: https://dashboard.ngrok.com/get-started/your-authtoken')
else:
    conf.get_default().auth_token = NGROK_TOKEN
    subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
    time.sleep(1)

    def run():
        subprocess.run(['streamlit', 'run', 'app.py',
                        '--server.port', '8501',
                        '--server.headless', 'true',
                        '--server.enableCORS', 'false',
                        '--server.enableXsrfProtection', 'false'])

    threading.Thread(target=run, daemon=True).start()
    time.sleep(5)
    url = ngrok.connect(8501)
    print(f'\n✅ App is live!')
    print(f'🌐 Open: {url}')
    print('\nKeep this cell running while using the app.')
